# Lab 20-09 · DACs

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import dac


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Ladders

*Predict* the output for code 683 and where a 1% switch resistance hurts most.

In [2]:
print("code 683:", dac.r2r_output(683, 10, 1.2), "V")
v = np.array([dac.r2r_output(c, 10, 1.2, r_sw=0.01) for c in range(1024)])
inl, dnl = dac.inl_dnl(v)
k = int(np.argmax(np.abs(dnl)))
print(f"1% switch resistance: worst DNL {dnl[k]:+.3f} LSB at transition {k}, worst INL {np.abs(inl).max():.3f} LSB; "
      f"step {1e3 * (v[k + 1] - v[k]):+.2f} mV (negative: non-monotonic)")
legs = np.array([dac.r2r_output(c, 10, 1.2, r_sw=0.01, mism_2r=np.r_[np.full(10, -0.005), 0.0]) for c in range(1024)])
print(f"legs made 2R - r_sw: worst |DNL| {np.abs(dac.inl_dnl(legs)[1]).max():.1e} LSB")
print(ngspice("r2r.cir"))
plt.figure(); plt.plot(inl); plt.xlabel("code"); plt.ylabel("INL (LSB)"); plt.show()

code 683: 0.8003906249999999 V
1% switch resistance: worst DNL -1.684 LSB at transition 511, worst INL 0.842 LSB; step -0.80 mV (negative: non-monotonic)
legs made 2R - r_sw: worst |DNL| 2.3e-13 LSB
{'vout': 0.8003906, 'dnlw': -1.68408, 'Stack': 0.0}


## Part 2 · Segmentation

*Predict* the spread of DNL at the worst transition for binary, unary and 6 + 8 segmented 14-bit DACs.

In [3]:
rng = np.random.default_rng(1)
for name, m in (("binary", 0), ("6 + 8 segmented", 6), ("unary", 14)):
    worst = []
    for _ in range(300 if m < 14 else 100):
        lv = dac.cs_levels(14, m, 0.005, rng)
        _, d = dac.inl_dnl(lv)
        b = 14 - m
        k = 2 ** 13 - 1 if m == 0 else (2 ** b - 1 if m < 14 else 2 ** 13 - 1)
        worst.append(d[k])
    print(f"{name:16s}: std of DNL at the worst step {np.std(worst):.3f} LSB")

binary          : std of DNL at the worst step 0.653 LSB
6 + 8 segmented : std of DNL at the worst step 0.106 LSB
unary           : std of DNL at the worst step 0.006 LSB


## Part 3 · Yield

*Predict* the fraction of 12-bit unary DACs with max |INL| ≤ ½ LSB at σ_u = 0.521% and 0.430%.

In [4]:
rng = np.random.default_rng(2)
for s in (0.00521, 0.00430):
    e = rng.normal(0, s, (4000, 4095))
    c = np.concatenate([np.zeros((4000, 1)), np.cumsum(1 + e, axis=1)], axis=1)
    inl = c / (c[:, -1:] / 4095) - np.arange(4096)
    print(f"sigma_u = {100 * s:.3f} %: yield {np.mean(np.abs(inl).max(axis=1) <= 0.5):.4f}")

sigma_u = 0.521 %: yield 0.9795


sigma_u = 0.430 %: yield 0.9978


## Part 4 · Output impedance

*Predict* the 15-cell DAC's full-scale output, then the single-ended and differential harmonics at xN = 0.0643.

In [5]:
print(ngspice("zout.cir"))
N, x = 4095, 0.0643 / 4095
se = lambda s: (N * (1 + s) / 2) / (1 + N * (1 + s) / 2 * x)
df = lambda s: se(s) - (N * (1 - s) / 2) / (1 + N * (1 - s) / 2 * x)
print("single-ended:", {k: round(v, 1) for k, v in dac.harmonics(se).items() if k > 1})
print("differential:", {k: round(v, 1) for k, v in dac.harmonics(df).items() if k > 1})

{'vfs': 0.7389163, 'inlmax': 0.05561072, 'Stack': 0.0}
single-ended: {2: -36.1, 3: -72.3, 4: -108.4, 5: -144.6}
differential: {2: -325.7, 3: -72.3, 4: -333.8, 5: -144.6}


## Part 5 · Output spectra and data-weighted averaging

*Predict* the NRZ, RZ and mixed-mode gains at 0.4, 0.5 and 0.6 fs; then predict how much DWA lowers the in-band mismatch error.

In [6]:
f = np.linspace(1e-4, 2, 2000)
plt.figure()
for mode in ("nrz", "rz", "mix"):
    plt.plot(f, 20 * np.log10(dac.zoh_gain(f, mode) + 1e-12), label=mode)
plt.ylim(-40, 2); plt.xlabel("f / fs"); plt.ylabel("dB"); plt.legend(); plt.show()
M = 16
elem = 1 + 0.02 * (np.arange(M) / (M - 1) - 0.5)
n, m = 4096, 41
codes = np.round(8 + 7.5 * np.sin(2 * np.pi * m * np.arange(n) / n)).astype(int)
therm = np.zeros((n, M), int)
for i, c in enumerate(codes):
    therm[i, :c] = 1
sig = np.abs(np.fft.rfft(codes - codes.mean())) ** 2
for name, sel in (("fixed selection", therm), ("DWA", dac.dwa_select(codes, M))):
    err = sel @ elem - codes * elem.mean()
    p = np.abs(np.fft.rfft(err)) ** 2
    print(f"{name:15s}: in-band error {10 * np.log10(p[1:n // 32 + 1].sum() / sig[m]):.1f} dB relative to the signal")

fixed selection: in-band error -52.5 dB relative to the signal
DWA            : in-band error -85.5 dB relative to the signal
